### `etd_euler.ipynb` 
*Created: May 7, 2026* <br/>

In this notebook we implement the Exponential Time-Differencing (ETD) method for solving ODE systems of the form 
\begin{align*}
    u_t = Au + f(u,t)
\end{align*}

where $A$ is an $N \times N$ constant matrix and $f:\mathbb{R}^{N} \times \mathbb{R} \to \mathbb{R}$ is some non-linear function. ETD methods are especially useful for ODEs of the above form, as the linear term $Au$ often makes the ODE system stiff; ETD handles this by "integrating the linear part exactly" (in a sense discussed below). More precisely, ETD makes use of the following *variation of parameters* formula: 
\begin{align*}
    \fbox{$u(t_{n+1}) = e^{\Delta t A} u_n + \int_{t_n}^{t_{n+1}} e^{(t_{n+1} - s)A} f(u(s),s)ds \,$}
\end{align*}

A fixed-step ETD scheme is just a scheme of the form 
\begin{align*}
   \qquad u_{n+1} = e^{\Delta t A} u_n + I_n, \qquad n = 0,1,2,\ldots
\end{align*}
where $I_n = I_n(u_n,\ldots,u_{n-k})$ is an approximation to the integral term. ETD Euler is obtained by making the approximation $f(u(\cdot),\cdot) \approx f(u(t_n), t_n))$ on $[t_{n},t_{n+1}]$: 
\begin{align*}
    I_n \approx \left(\int_{t_n}^{t_{n+1}} e^{(t_{n+1} - s)A}\right) f(u_n,t_n) = \left(\int_{0}^{\Delta t} e^{sA} \,ds \right) f(u_n,t_n). 
\end{align*}

The last integral can be expressed as a series: 
\begin{align*}
    \int_{0}^{\Delta t} e^{sA} \,ds = \int_{0}^{\Delta t} \sum_{k=0}^{\infty} \frac{(sA)^k}{k!} \,ds = \sum_{k=0}^{\infty} \int_{0}^{\Delta t}  \frac{(sA)^k}{k!} \,ds = \sum_{k=0}^{\infty} \frac{(\Delta t)^{k+1}A^k}{(k+1)!} = \Delta t \underbrace{\left(\sum_{k=0}^{\infty} \frac{(\Delta t A)^k}{(k+1)!}\right)}_{=:\varphi_1(\Delta t A)}
\end{align*}

The last series in the above line is the definition for the matrix-valued function $\varphi_1: \mathbb{R}^{n \times n} \to \mathbb{R}^{n \times n}$, which comes up frequently in ETD schemes (along with other so-called *phi-functions*). Thus, 
\begin{align*}
   I_n \approx \varphi_1(\Delta t A). 
\end{align*}

Noting that $e^{\Delta t A} = \varphi_0(\Delta t A)$, the ETD-Euler scheme becomes
\begin{align*}
   \textbf{ETD Euler:} \quad \fbox{$\displaystyle u_{n+1} = \varphi_0(\Delta t A) u_n + \Delta t \, \varphi_1(\Delta t A) f(u_n,t_n)$}
\end{align*}

<div><strong>Remarks:</strong></div>
<ul style="margin-top: 0;">
  <li>The matrices $\varphi_0(\Delta t A)$ and $\varphi_1(\Delta t A)$ are computed *once* at the outset, making this method very cheap.  </li>
</ul>

In [1]:
using OrdinaryDiffEq, LaTeXStrings, Printf, UnPack, NBInclude, LinearAlgebra, Random, FFTW, ExponentialUtilities
@nbinclude("../../../phi_functions/phi_functions.ipynb");

In [2]:
function etd_euler(A, f, u0, tspan::NTuple{2,Float64}, p = nothing; dt::Float64)
    """
    First-order Exponential Time Differencing (ETD) method for solving the semi-linear ODE 
    
                                    u_t = Au + f(u,p,t)  on [t0, tf]. 

    Here, u: [t0,tf] -> R^N is the ODE solution, A is an N x N constant matrix, f is a nonlinear function from R^N × [t0,tf] to R^N
    The function definition for `f` must include `p` as an argument, whether or not it is actually used. 

    Notes:
    - The solution is computed at the evenly spaced points t_0, t_0 + dt, t_0 + 2dt,...,t_0 + num_steps*dt
      where num_steps := floor((tf - t0) / dt). 
    - The last time point is t_last = t_0 + N*dt. Note that t_last ≤ tspan[2], in general, with equality
      if and only if (tf - t0) / dt is an integer. 
    - f(u,p,t) must be either a vector or a scalar matching the dimension of `u0`.
    
    
    PARAMETERS
    ----------
    A :: N x N matrix (or a scalar) 
    f :: a function `f(u,p,t)` from R^N to R (the non-linear term) where u is the state, p is a scalar, and t is time. 
    u0 :: the initial condition; a scalar (if N = 1) or a vector (if N > 1) 
    tspan :: time interval over which to integrate (a tuple) 
    p :: parameter tuple for the nonlinear term (if required) 
    dt :: the time step 
    
    RETURNS
    -------
    sol.u :: vector of solution iterates 
    sol.t :: time values at which the solution was computed 
    sol.p :: the ODE parameter (a scalar, a tuple, or nothing)
    sol.dt :: the time step 
    """

    #Input validation
    all(isfinite, tspan) || throw(ArgumentError("Time interval must be finite."))
    tspan[1] < tspan[2] || throw(ArgumentError("`tspan[1]` must be strictly less than `tspan[2]`."))
    isfinite(dt) || throw(ArgumentError("dt must be finite."))
    dt > 0 || throw(ArgumentError("dt must be strictly positive."))

    #Compute number of steps and time points
    t0, tf = tspan                                             #Endpoints of time interval
    num_steps = Int(floor((tf - t0)/dt))                       #Number of steps to be taken by the ODE solver 
    t = collect(range(t0, step = dt, length = num_steps + 1))  #Time values at which to record the solution 
                                                            
    #Initialize array to store solution
    u0 = float.(u0)   
    u = [zero(u0) for _ in 1:num_steps + 1]            
    u[1] = copy(u0)                                   

    #Compute phi matrices (ϕ₀ and ϕ₁ are defined in `phi_contour.ipynb`; uses M = 64 quadrature points)
    Φ₀ = phi(dt*A, 0) 
    Φ₁ = phi(dt*A, 1)
      
    for n = 1:num_steps     
        u[n+1] = Φ₀ * u[n] + dt * (Φ₁ * f(u[n], p, t[n]))
    end 

    return (u = u, t = t, p = p, dt = dt) 
end 

etd_euler (generic function with 2 methods)